# Conversational RAG — Bramblewood Gear Co. / FieldWise

**Company:** Bramblewood Gear Co., an outdoor equipment retailer.
**Product:** FieldWise, the internal assistant support agents use while
working a ticket.

This notebook covers a conversational retrieval workflow for a support
ticket that spans several turns:

- chat history + retrieval
- LangGraph-style stateful RAG
- routing (cache vs. retrieval vs. direct reply)
- semantic caching for repeated FAQ-style questions
- per-ticket session memory, plus longer-term customer memory

Short-term memory here is scoped to one ticket (thread), while long-term
memory persists customer preferences across tickets. A router decides
whether an agent's question needs a cache hit, a fresh retrieval, or a
direct reply, and the final answer is generated by an LLM (with an offline
fallback -- see Section 8) grounded in whatever was retrieved.

## Learning goals

By the end of this notebook, you should be able to:

1. Keep chat history as part of the retrieval process.
2. Route a request to retrieval, cache, or direct answer logic.
3. Store per-session (per-ticket) conversation state.
4. Add a simple semantic cache.
5. Generate a grounded reply with an LLM, with a deterministic offline
   fallback when one isn't reachable.

## 1) Install packages

```bash
pip install -U scikit-learn langchain langchain-community langchain-text-splitters langchain-core langchain-groq faiss-cpu python-dotenv
```

In [ ]:
%pip install -qU scikit-learn langchain langchain-community langchain-text-splitters langchain-core langchain-groq faiss-cpu python-dotenv

## 2) Load environment variables

Create a `.env` file with:

```env
GROQ_API_KEY=your_groq_key
GROQ_MODEL=llama-3.3-70b-versatile
```

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")

print("GROQ_API_KEY set:", bool(GROQ_API_KEY))
print("GROQ_MODEL:", GROQ_MODEL)

## 3) Bramblewood's support knowledge base

The same kind of gear-care and policy content FieldWise searches over
elsewhere in this module.

In [ ]:
from langchain_core.documents import Document

knowledge_docs = [
    Document(
        page_content=(
            "Tent rainflies use a factory-applied waterproof coating that "
            "wears down with UV exposure. If a rainfly leaks, reapply a DWR "
            "treatment and reseal the seams with seam sealer."
        ),
        metadata={"source": "tent_care", "topic": "tents"},
    ),
    Document(
        page_content=(
            "Returns are accepted within 60 days for unused gear in original "
            "packaging. Used gear is handled through a warranty claim "
            "instead: 2 years on tents and stoves, lifetime on backpack "
            "stitching."
        ),
        metadata={"source": "returns_warranty", "topic": "policy"},
    ),
    Document(
        page_content=(
            "The Ember Down sleeping bag loses loft if washed with regular "
            "detergent. Wash with technical down soap on a gentle cycle and "
            "tumble dry low with clean tennis balls to restore loft."
        ),
        metadata={"source": "sleeping_bag_care", "topic": "sleeping_bags"},
    ),
    Document(
        page_content=(
            "Basecamp backpack zipper sliders can stick after heavy use. Dry "
            "silicone lubricant usually frees a stiff slider; a broken "
            "slider is covered under the lifetime workmanship warranty."
        ),
        metadata={"source": "backpack_zippers", "topic": "backpacks"},
    ),
]

knowledge_docs

## 4) Split and embed the documents

We check for a live embedding model and fall back to an offline one if it
isn't reachable, same as the rest of this module.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.embeddings import Embeddings
from sklearn.feature_extraction.text import HashingVectorizer


class OfflineHashingEmbeddings(Embeddings):
    """Deterministic, dependency-free embedding fallback (see the
    end-to-end pipeline notebook for the full rationale)."""

    def __init__(self, n_features: int = 512):
        self.n_features = n_features
        self._hv = HashingVectorizer(
            n_features=n_features, alternate_sign=False, norm="l2", stop_words="english"
        )

    def embed_documents(self, texts):
        return self._hv.transform(texts).toarray().tolist()

    def embed_query(self, text):
        return self._hv.transform([text]).toarray()[0].tolist()


def get_embeddings():
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
        candidate = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        candidate.embed_query("connectivity check")
        print("Embeddings: live sentence-transformers model loaded (semantic).")
        return candidate, "live"
    except Exception as e:
        print(f"Embeddings: falling back to offline hashing embeddings -- {type(e).__name__}: {str(e)[:160]}")
        return OfflineHashingEmbeddings(), "offline"


embeddings, EMBEDDINGS_MODE = get_embeddings()

splitter = RecursiveCharacterTextSplitter(chunk_size=140, chunk_overlap=20)
chunks = splitter.split_documents(knowledge_docs)

vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

print("Chunks:", len(chunks), "| Embeddings mode:", EMBEDDINGS_MODE)

## 5) Basic chat history + retrieval

The current message should be interpreted together with the ticket's chat
history so far. A simple way to do this:
- keep a list of messages
- fold recent history into the retrieval query
- retrieve relevant context
- answer using both the history and retrieved context

In [ ]:
def get_last_user_message(messages):
    for msg in reversed(messages):
        if msg["role"] == "user":
            return msg["content"]
    return ""

def build_retrieval_query(messages):
    last_user = get_last_user_message(messages)
    history_snippet = []
    for msg in messages[-4:]:
        history_snippet.append(f"{msg['role']}: {msg['content']}")
    return last_user + " Conversation context: " + " ".join(history_snippet)

messages = [
    {"role": "user", "content": "Customer's Summit Ridge tent is leaking overnight, what should I tell them?"},
]

retrieval_query = build_retrieval_query(messages)
print(retrieval_query)

results = retriever.invoke(retrieval_query)
for i, doc in enumerate(results, 1):
    print(f"--- Retrieved {i} ---")
    print(doc.page_content)
    print(doc.metadata)

## 6) LangGraph-style stateful RAG

LangGraph treats the state object as short-term memory during a run. This
notebook uses a plain Python dictionary to mirror that idea without
building a full graph.

In [ ]:
state = {
    "messages": [],
    "retrieved_docs": [],
    "route": None,
    "ticket_id": "ticket-4471",
}

def update_state(state, role, content):
    state["messages"].append({"role": role, "content": content})

update_state(state, "user", "Customer's Summit Ridge tent is leaking overnight, what should I tell them?")
state["retrieved_docs"] = retriever.invoke(build_retrieval_query(state["messages"]))
state["route"] = "retrieval"

state

## 7) Routing

A router chooses where the request should go. For FieldWise, useful routes
are:

- `cache` -- a question that has already been answered recently
- `retrieval` -- a question about a specific product, care, or policy topic
- `direct` -- small talk that doesn't need the knowledge base

In [ ]:
def route_query(question: str) -> str:
    q = question.lower()

    if any(word in q for word in ["again", "same", "repeat", "asked this before"]):
        return "cache"

    if any(word in q for word in [
        "tent", "leak", "rainfly", "waterproof", "warranty", "return",
        "sleeping bag", "backpack", "zipper", "stove",
    ]):
        return "retrieval"

    return "direct"

tests = [
    "Customer's tent rainfly is leaking, what do I tell them?",
    "What's our return policy again?",
    "Thanks, that's everything for this ticket.",
]

for q in tests:
    print(q, "->", route_query(q))

## 8) Session memory

Short-term memory is scoped to a single ticket, while long-term memory
persists across tickets for the same customer. We model per-ticket memory
with a simple in-memory store keyed by `ticket_id`.

In [ ]:
from collections import defaultdict

session_memory = defaultdict(list)

def append_session_message(ticket_id: str, role: str, content: str):
    session_memory[ticket_id].append({"role": role, "content": content})

def get_session_history(ticket_id: str):
    return session_memory[ticket_id]

append_session_message("ticket-4471", "user", "Customer's Summit Ridge tent is leaking overnight, what should I tell them?")

get_session_history("ticket-4471")

## 9) Simple semantic caching

Semantic caching reuses an answer when a new question is close in meaning
to one that was already answered -- useful for the handful of questions
("what's your return policy?") that come up on nearly every ticket.

In [ ]:
import numpy as np

cache_questions = []
cache_answers = []
cache_vectors = []

def embed_text(text: str):
    return np.array(embeddings.embed_query(text), dtype=float)

def cosine_sim(a, b):
    denom = (np.linalg.norm(a) * np.linalg.norm(b))
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)

def cache_lookup(question: str, threshold: float = 0.6):
    if not cache_questions:
        return None, 0.0

    q_vec = embed_text(question)
    best_idx = -1
    best_score = -1.0

    for i, cached_vec in enumerate(cache_vectors):
        score = cosine_sim(q_vec, cached_vec)
        if score > best_score:
            best_score = score
            best_idx = i

    if best_score >= threshold:
        return cache_answers[best_idx], best_score

    return None, best_score

def cache_store(question: str, answer: str):
    cache_questions.append(question)
    cache_answers.append(answer)
    cache_vectors.append(embed_text(question))

cache_store(
    "What's Bramblewood's return policy?",
    "Unused gear can be returned within 60 days in original packaging; used gear needs a warranty claim instead.",
)
cached_answer, score = cache_lookup("What is the return policy?")
print("Cached answer:", cached_answer)
print("Similarity score:", score)

## 10) LLM availability check

The final reply is generated by an LLM grounded in whatever was retrieved.
We check once whether Groq is reachable and fall back to a small
deterministic mock if it isn't, so the rest of the notebook runs the same
way either way.

In [ ]:
import re
from typing import Any, List, Optional

from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.prompts import ChatPromptTemplate


def _extract_section(text: str, label: str) -> str:
    """Pull a labeled section like 'Context:' out of a rendered prompt
    string, stopping at the next 'Label:' marker (or the end of the
    string) so it doesn't grab everything that follows."""
    pattern = rf"{re.escape(label)}:\s*(.*?)(?=\n\s*[A-Z][a-zA-Z ]{{0,20}}:|\Z)"
    match = re.search(pattern, text, flags=re.DOTALL)
    return match.group(1).strip() if match else ""


class OfflineMockChatModel(BaseChatModel):
    """Deterministic offline stand-in for ChatGroq: an extractive heuristic
    that keeps LCEL-style chains running end-to-end without a live Groq
    connection. Every response is prefixed with "[offline mock]"."""

    @property
    def _llm_type(self) -> str:
        return "offline-mock-chat-model"

    def _generate(self, messages: List[BaseMessage], stop: Optional[List[str]] = None, run_manager: Optional[Any] = None, **kwargs: Any) -> ChatResult:
        human_text = " ".join(m.content for m in messages if m.type == "human" and isinstance(m.content, str))
        context = _extract_section(human_text, "Context") or human_text
        question = _extract_section(human_text, "Question") or human_text
        answer = self._extractive_answer(question, context)
        message = AIMessage(content=f"[offline mock] {answer}")
        return ChatResult(generations=[ChatGeneration(message=message)])

    @staticmethod
    def _extractive_answer(question: str, context: str, max_sentences: int = 2) -> str:
        q_terms = {w.lower() for w in re.findall(r"[a-zA-Z]{4,}", question)}

        chunk_blocks = [c.strip() for c in re.split(r"\n\s*\n", context.strip()) if c.strip()]
        if not chunk_blocks:
            chunk_blocks = [context.strip()]

        def term_overlap(text: str) -> int:
            text_terms = {w.lower() for w in re.findall(r"[a-zA-Z]{4,}", text)}
            return len(q_terms & text_terms)

        best_chunk = max(chunk_blocks, key=term_overlap)

        sentences = [
            s for s in re.split(r"(?<=[.!?])\s+", best_chunk.strip())
            if len(s.strip()) > 15
        ]
        scored = []
        for s in sentences:
            if s.strip():
                scored.append((term_overlap(s), s.strip()))
        scored.sort(key=lambda x: x[0], reverse=True)
        top = [s for _, s in scored[:max_sentences]] or (sentences[:1] if sentences else [""])
        return " ".join(top)[:600]


def get_llm():
    try:
        from langchain_groq import ChatGroq
        candidate = ChatGroq(model=GROQ_MODEL, temperature=0)
        candidate.invoke("connectivity check")
        print("LLM: live Groq model connected.")
        return candidate, "live"
    except Exception as e:
        print(f"LLM: falling back to offline mock chat model -- {type(e).__name__}: {str(e)[:160]}")
        return OfflineMockChatModel(), "offline"


llm, LLM_MODE = get_llm()

answer_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are FieldWise, Bramblewood's internal support assistant. Answer "
        "using only the provided context, in 2-3 sentences the agent can "
        "paste straight into a reply. If the context doesn't cover it, say so."
    ),
    ("user", "Question: {question}\n\nContext:\n{context}"),
])

print("answer_prompt ready. LLM mode:", LLM_MODE)

## 11) A basic conversational RAG flow

The flow is:

1. Check the cache.
2. Route the request.
3. If needed, build a retrieval query from ticket history.
4. Retrieve documents.
5. Ask the LLM to phrase the answer from that context.
6. Save the result into session memory and cache.

In [ ]:
def conversational_rag_answer(ticket_id: str, user_question: str):
    append_session_message(ticket_id, "user", user_question)

    cached_answer, score = cache_lookup(user_question)
    if cached_answer:
        route = "cache"
        answer = cached_answer
        retrieved = []
    else:
        route = route_query(user_question)

        if route == "retrieval":
            history = get_session_history(ticket_id)
            retrieval_query = build_retrieval_query(history)
            retrieved = retriever.invoke(retrieval_query)
            context = "\n\n".join(doc.page_content for doc in retrieved)
            prompt_messages = answer_prompt.format_messages(question=user_question, context=context)
            response = llm.invoke(prompt_messages)
            answer = response.content
        else:
            retrieved = []
            answer = (
                "Happy to help -- let me know the product and what's going "
                "on and I'll pull up the relevant care or warranty info."
            )

        cache_store(user_question, answer)

    append_session_message(ticket_id, "assistant", answer)
    return {
        "route": route,
        "answer": answer,
        "retrieved_count": len(retrieved),
    }

result = conversational_rag_answer("ticket-4471", "Customer's Summit Ridge tent is leaking overnight, what should I tell them?")
result

## 12) Follow-up question behavior

The main value of conversational RAG is that follow-up questions can reuse
the ticket's history. A follow-up like "what about just the rainfly?" can
still be grounded in the earlier turn instead of being retrieved cold.

In [ ]:
follow_up = conversational_rag_answer("ticket-4471", "They're asking specifically about the rainfly, not the floor.")
follow_up

## 13) Long-term memory

Long-term memory is where you'd store customer preferences or durable
account facts across tickets -- which product they own, how they like to
be contacted. LangChain's long-term memory is built on LangGraph stores; we
keep it conceptual here with a plain dictionary.

In [ ]:
long_term_memory = {}

def remember_customer_fact(customer_id: str, key: str, value: str):
    long_term_memory.setdefault(customer_id, {})
    long_term_memory[customer_id][key] = value

def recall_customer(customer_id: str):
    return long_term_memory.get(customer_id, {})

remember_customer_fact("customer-8821", "product_owned", "Summit Ridge 2P Tent")
remember_customer_fact("customer-8821", "preferred_channel", "email")

recall_customer("customer-8821")

## 14) Tool-wrapper routing pattern

Wrapping the router as a tool keeps routing logic stateless and separate
from memory and context, which a conversational agent can hold onto
instead.

In [ ]:
def router_tool(query: str) -> str:
    route = route_query(query)
    if route == "retrieval":
        return "Would call FieldWise's retrieval workflow"
    if route == "cache":
        return "Would return a cached answer"
    return "Would answer directly"

for q in tests:
    print(q, "->", router_tool(q))

## 15) What LangGraph adds

LangGraph is useful when you want:
- persistent state
- multi-turn memory
- branching paths
- explicit routing
- checkpoint-based conversation continuity

For a real FieldWise deployment, this notebook's plain dictionaries would
become LangGraph state and a LangGraph store, which is what the
`end_to_end_rag_pipeline` and `mcp_core_concepts_langchain` notebooks in
this module build toward.

## Key takeaways

- Short-term memory is thread- (ticket-) scoped state.
- Long-term memory persists across tickets for the same customer.
- Chat history should influence retrieval for follow-up questions.
- Routing can decide whether to use cache, retrieval, or a direct reply.
- Semantic caching reduces repeated retrieval + generation work for
  FAQ-style questions.
- The LLM call is grounded in retrieved context, with a deterministic
  offline fallback so the flow runs the same way with or without a live
  Groq connection.

## References

- Short-term memory: https://docs.langchain.com/oss/python/langchain/short-term-memory
- Long-term memory: https://docs.langchain.com/oss/python/langchain/long-term-memory
- Agentic RAG: https://docs.langchain.com/oss/python/langgraph/agentic-rag
- Router tool wrapper: https://docs.langchain.com/oss/python/langchain/multi-agent/router
- LangSmith caching: https://docs.langchain.com/langsmith/caching
- Prompt caching: https://docs.langchain.com/langsmith/manage-prompts-programmatically

## Verification summary

| Section | Status | Notes |
|---|---|---|
| Package install | Executed | pip install cell |
| Knowledge base + chunking | Executed & verified | in-memory documents |
| Embeddings | Executed & verified | see printed `EMBEDDINGS_MODE` |
| Chat history / state / routing / session memory / cache | Executed & verified | pure Python, deterministic |
| LLM-backed answer generation | Executed & verified | see printed `LLM_MODE` -- offline mock in a sandbox with no Groq access, live Groq on a machine with `GROQ_API_KEY` set and network access |
| Follow-up + long-term memory | Executed & verified | reuses the same session/cache/LLM plumbing above |

On a machine with a working `GROQ_API_KEY`, `LLM_MODE` will read `live` and
`conversational_rag_answer` will return real generated replies instead of
the offline extractive mock.